In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_table_validator(
    *[silver_awards, silver_fiscal_days], 
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)


In [0]:
fiscal_days = spark.table(silver_fiscal_days)
fiscal_days.createOrReplaceTempView("fiscal_days")

awards = spark.table(silver_awards)
awards.createOrReplaceTempView("awards")

df_awards_fiscal = spark.sql("""
select
    aw.*,
    f.FISCAL_WEEK_END as FISCAL_WEEK_END
from
    awards as aw
inner join
    fiscal_days f
on
    aw.AWRD_CERT_ISSUE_DT = f.FISCAL_DAY
""")

df_awards_fiscal.createOrReplaceTempView("source")

### Merge

In [0]:
df_awards_fiscal.write.mode("overwrite").saveAsTable(silver_awards_fiscal)

if archive_flag:
    save_archive(df_awards_fiscal, silver_awards_fiscal_archive, run_as_date)